## Notebook 05 - EDA FINAl, Consultas a MySQL

Ultimo notebook, donde preparo todo el camino para Streamlit.
No quiero pasar millones de datos al dashboard, solo los datos de las consultas pertinentes.

In [12]:
import pandas as pd
import sys
import plotly.express as px
from pathlib import Path
from sqlalchemy import text


# Añado config a la esta ruta
sys.path.append(str(Path.cwd().parent))

from src.config import PROCESSED_DATA_DIR
from src.database_mgr import get_engine

# inicializo el motor de la base de datos
engine = get_engine()

In [13]:
def get_severity_metrics():
    """
    Calcula la tasa de hospitalizacion y letalidad.
    """
    query = text("""
    SELECT
        SUM(num_casos) as total_casos,
        SUM(num_def) as total_muertes,
        (SUM(num_def) / SUM(num_casos)) * 100 as tasa_mortalidad,
        (SUM(num_uci) / SUM(num_hosp)) * 100 as ratio_uci_hosp
        FROM daily_stats
    """)
    with engine.connect() as conn:
        df= pd.read_sql(query, conn)
    return df

stats = get_severity_metrics()
print(f"Tasa de mortalidad media: {stats["tasa_mortalidad"][0]:.2f}%")

Tasa de mortalidad media: 0.95%


In [14]:
def get_mortality_by_province():
    """
    Calcula la tasa de mortalidad por cada provincia_iso.
    """
    query = text("""
    SELECT
        provincia_iso,
        SUM(num_casos) as casos,
        SUM(num_def) as muertes,
        (SUM(num_def) / SUM(num_casos)) * 100 as tasa_mortalidad
        FROM daily_stats
        GROUP BY provincia_iso
        HAVING casos > 1000
        ORDER BY tasa_mortalidad DESC
        """)
    with engine.connect() as conn:
        return pd.read_sql(query, conn)

df_mortalidad = get_mortality_by_province()
print(df_mortalidad.head(10))

  provincia_iso     casos  muertes  tasa_mortalidad
0            SO   29411.0    564.0             1.92
1            TO  193572.0   3150.0             1.63
2            ZA   50811.0    814.0             1.60
3            AV   48051.0    758.0             1.58
4            CR  134742.0   2119.0             1.57
5            SG   46721.0    698.0             1.49
6             O  242913.0   3560.0             1.47
7            TE   44285.0    617.0             1.39
8            CU   60766.0    829.0             1.36
9            SA  103705.0   1364.0             1.32


In [17]:
def get_daily_evolution():
    query = text("""
        SELECT fecha, SUM(num_casos) as casos_diarios,
        SUM(num_hosp) as hosp_diarios
        FROM daily_stats
        GROUP BY fecha
        ORDER BY fecha ASC
        """)
    with engine.connect() as conn:
        return pd.read_sql(query, conn)

df_evolucion = get_daily_evolution()
fig = px.line(df_evolucion, x="fecha", y="casos_diarios", title="Evolución Diaria de Casos en España")
fig.show()

In [ ]:
def get_daily_evolution_filtered(provincia_code = None):
    if provincia_code is None:
        query = text("""
            SELECT fecha, SUM(num_casos) as casos_diarios,
            SUM(num_hosp) as hosp_diarios
            FROM daily_stats
            GROUP BY fecha
            ORDER BY fecha ASC
    """)
        params = {}
    else:
        query = text("""
        SELECT fecha, num_casos as casos
        FROM daily_stats
        WHERE provincia_iso = :p ORDER BY fecha""")
        params = {'p': provincia_code}
    with engine.connect() as conn:
        return pd.read_sql(query, conn, params=params)

df_madrid = get_daily_evolution_filtered("M")
print(df_madrid.head(10))